# `sparsegf2.circuits.graphs` - interaction geometry

A `GraphTopology` stores the allowed two-qubit edges, graph6 provenance, and
matching information used by brickwork schedules. The scheduler decides which
allowed edges fire. Graph construction itself does not touch the simulator.

`from_spec` supports six string families:

1. `cycle`
2. `complete`
3. `path`
4. `lattice_2d`
5. `newman_watts`
6. `watts_strogatz`

The two small-world families accept inline parameters. `from_networkx` adapts
any simple undirected NetworkX graph with arbitrary original node labels.


In [1]:
from sparsegf2.circuits import from_spec

specs = [
    'cycle', 'complete', 'path', 'lattice_2d',
    'newman_watts(k=2,p=0.2)',
    'watts_strogatz(k=2,beta=0.25)',
]
graphs = [from_spec(spec, 16, seed=11) for spec in specs]
for spec, graph in zip(specs, graphs, strict=True):
    print(
        f'{spec:<35} |E|={len(graph.edges):>3}  '
        f'stochastic={graph.is_stochastic!s:<5}  graph6 chars={len(graph.graph6)}'
    )


cycle                               |E|= 16  stochastic=False  graph6 chars=21
complete                            |E|=120  stochastic=False  graph6 chars=21
path                                |E|= 15  stochastic=False  graph6 chars=21
lattice_2d                          |E|= 24  stochastic=False  graph6 chars=21
newman_watts(k=2,p=0.2)             |E|= 19  stochastic=True   graph6 chars=21
watts_strogatz(k=2,beta=0.25)       |E|= 32  stochastic=True   graph6 chars=21


## Matchings and gate schedules

`brickwork` requires a matching. `round_robin` and `palette` consume a stored
1-factorization; `fresh` draws a perfect matching when the geometry supplies a
sampler. The other gate schedules need only a nonempty edge set:
`random_edge` draws distinct edges, `random_pool` draws with replacement, and
`all_edges` fires every stored edge in order.

Irregular or stochastic graphs generally do not have a fixed
1-factorization, but remain valid with edge-based schedules.


In [2]:
cycle = from_spec('cycle', 8)
complete = from_spec('complete', 8)
print('C8 factors / edges:', cycle.chi_prime, '/', len(cycle.edges))
print('K8 factors / edges:', complete.chi_prime, '/', len(complete.edges))
assert sorted(edge for factor in cycle.one_factorization for edge in factor) == cycle.edges
print('cycle factors partition its edge set:', True)


C8 factors / edges: 2 / 8
K8 factors / edges: 7 / 28
cycle factors partition its edge set: True


## Stochastic realization seeds

For a stochastic string specification, `CircuitConfig.base_seed` is the
quenched graph-realization seed. Reusing `(specification, n, seed)` reconstructs
the same sorted edge list and graph6 string. Changing `sample_seed` later changes
the circuit trajectory without changing this geometry.


In [3]:
spec = 'watts_strogatz(k=2,beta=0.25)'
a = from_spec(spec, 32, seed=7)
b = from_spec(spec, 32, seed=7)
c = from_spec(spec, 32, seed=8)
print('same realization seed:', a.edges == b.edges)
print('different realization seed:', a.edges != c.edges)
print('edge count and mean degree:', len(a.edges), 2 * len(a.edges) / a.n)


same realization seed: True
different realization seed: True
edge count and mean degree: 64 4.0


## Arbitrary NetworkX geometry

`from_networkx` relabels nodes to `0, ..., n-1`, rejects multigraphs and
self-loops, canonicalizes the edge list, and records graph6 provenance.
`CircuitConfig` also accepts the NetworkX object directly and performs this
adaptation during validation.


In [4]:
import networkx as nx
from sparsegf2.circuits import CircuitBuilder, CircuitConfig, from_networkx

raw = nx.wheel_graph(8)
adapted = from_networkx(raw, name='wheel8')
cfg = CircuitConfig(
    graph_spec=raw, n=8, gating_mode='random_edge',
    total_layers_override=2,
)
print(adapted.name, '|E|=', len(adapted.edges), 'graph6=', adapted.graph6)
print('direct NetworkX config resolves to:', CircuitBuilder(cfg, 0).graph.name)


wheel8 |E|= 14 graph6= G|eKMC
direct NetworkX config resolves to: networkx(8n_14e)


## Summary

The six named families and `from_networkx` share one canonical
`GraphTopology` representation. Deterministic graphs can expose reusable
1-factorizations; stochastic graphs preserve their exact realization through
the seed and graph6 metadata.
